<a href="https://colab.research.google.com/github/shreyanalam/RAG-PDF-Chatbot/blob/main/AI_Weather_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# AI code review
import google.generativeai as genai
import gradio as gr

genai.configure(api_key=" ")
def review_code(code,input_programming_language,output_programming_languages):
  prompt=f"""
  Review this code from input_programming_language to output_programming_languages.
  provide:
  1.Summary
  2.Bugs
  3.Performance Improvements
  4.Code style
  5.Security issues
  6.Improved code
  7.Rating out of 10
  Code:
  {code}
  Input Programming Language:
  {input_programming_language}
  Output Programming Language:
  {output_programming_languages}
  """
  model = genai.GenerativeModel("gemini-2.5-flash")
  response = model.generate_content(prompt)
  return response.text
demo=gr.Interface(
    fn=review_code,
    inputs=[
    gr.Textbox(label="Code"),
    gr.Dropdown(
        ["python","java","html","css"],
        label="input_programming_language"
    ),
    gr.Dropdown(
        ["python","java","html","css"],
        label="output_programming_language"
    ),
    ],
    outputs="markdown",
    title=" AI Code Review (Gemini)"
)
demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://361eb86f8a2f7a6c28.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



KeyboardInterrupt



In [ ]:
# gen ai tutor
from google import genai
import gradio as gr
client=genai.Client(api_key=" ")
SYSTEM_PROMPT="""
  You are a studymate AI and very intelligent tutor and educational assistant
  Instructions:
- Answer study-related questions clearly and accurately.
- Explain concepts in simple and easy-to-understand language.
- Provide step-by-step explanations whenever appropriate.
- Use examples to improve understanding.
- If programming code is requested, provide complete and well-commented code.
- Explain code line by line if requested.
- Present information using headings, bullet points, or tables whenever helpful.
- If solving numerical problems, show all calculation steps.
- If multiple solutions exist, explain the best one first.
- If you are unsure of an answer, clearly state that instead of guessing.
- Maintain a polite, friendly, and professional tone.
- Format every response neatly for easy reading.
"""
chat=client.chats.create(
    model="gemini-2.5-flash",
    config={
        "system_instruction":SYSTEM_PROMPT
    }
)

def respond(message,history):
  global chat
  response=chat.send_message(message)
  return response.text
def clear_chat():
  global chat

  chat=client.chats.create(
      model="gemini-2.5-flash",
      config={
          "system_instruction":SYSTEM_PROMPT
      }
  )

  return [],""
with gr.Blocks(title="AI STUDY TUTORR") as demo:
  gr.Markdown("# study Mate AI")
  gr.Markdown("# Ask any study related question")
  chatbot=gr.ChatInterface(
      fn=respond,
  )
  clear_btn=gr.Button("Clear Chat")
  clear_btn.click(
      fn=clear_chat,
      outputs=[chatbot.chatbot,chatbot.textbox]
  )
  demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://48e3c51e3936a32bbc.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
#weather agent
!pip install -q gradio google-genai requests
import gradio as gr
import requests
from google import genai
client=genai.Client(api_key=" ")
def get_coordinates(city):
  url=f"https://geocoding-api.open-meteo.com/v1/search?name={city}&count=1"
  response=requests.get(url,timeout=10)
  data=response.json()
  if "results" not in data:
    return None
  lat=data["results"][0]["latitude"]
  lon=data["results"][0]["longitude"]
  return lat,lon
def get_weather(lat,lon):
  url=(
      f"https://api.open-meteo.com/v1/forecast?"
      f"latitude={lat}&longitude={lon}"
      f"&current=temperature_2m,relative_humidity_2m,"
      f"wind_speed_10m,weather_code"
  )
  response=requests.get(url,timeout=10)
  data=response.json()
  return data["current"]
def weather_agent(query):
  prompt=f"""
extract only the city name from sentence below.
Sentence:
{query}
Return only the city name.
"""
  city=client.models.generate_content(
      model="gemini-2.5-pro",
      contents=prompt
  ).text.strip()
  location=get_coordinates(city)
  if location is None:
    return "City not found."
  lat,lon=location
  weather=get_weather(lat,lon)
  final_prompt=f"""
The user asked:
{query}
Weather Data
temperature:{weather['temperature_2m']}c
humidity:{weather['relative_humidity_2m']}%
wind_speed:{weather['wind_speed_10m']}km/h
weather_code:{weather['weather_code']}
Explain the weather in a friendly way
"""
  answer=client.models.generate_content(
      model="gemini-2.5-pro",
      contents=final_prompt
  )
  return answer.text

demo = gr.Interface(
    fn=weather_agent,
    inputs=gr.Textbox(
        label="Ask About Weather",
        placeholder="Example: What's the weather in Hyderabad?"
    ),
    outputs=gr.Textbox(label="Weather Report"),
    title="🌦️ AI Weather Agent",
    description="Ask about the weather in any city."
)

demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://baf07d019288e2972d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
#rag
!pip install -q gradio chromadb pypdf sentence-transformers google-generativeai
import gradio as gr
import chromadb
import google.generativeai as genai
import uuid
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
genai.configure(api_key=" ")
llm=genai.GenerativeModel("gemini-2.5-flash")
print("Loading embedding model...")
embedder=SentenceTransformer("all-MiniLM-L6-v2")
print("Embedding model loaded!")
client=chromadb.Client()
collection=client.get_or_create_collection("rag_docs")
def extract_text(pdf):
  reader=PdfReader(pdf.name)
  text = ""
  for page in reader.pages:
    page_text=page.extract_text()
    if page_text:
      text+=page_text+"\n"
  return text
def chunk_text(text,size=300):
  return [text[i:i+size] for i in range(0,len(text),size)]
def upload_pdf(pdf):
  global collection
  text=extract_text(pdf)
  chunks=chunk_text(text)
  print(f"Chunks Created: {len(chunks)}")
  embeddings=embedder.encode(chunks).tolist()
  #reset collection for new PDF
  try:
    client.delete_collection("rag_docs")
  except:
    pass
  collection=client.get_or_create_collection("rag_docs")
  collection.add(
      documents=chunks,
      embeddings=embeddings,
      ids=[str(uuid.uuid4())for _ in chunks]
      )
  return f" PDF Indexed Successfully ({len(chunks)} chunks)"
def ask(question):
  query_embedding=embedder.encode(question).tolist()
  result=collection.query(
      query_embeddings=[query_embedding],
      n_results=3
  )
  context="\n".join(result["documents"][0])
  context=context[:4000]
  prompt=f"""
Answer ONLY using the context below
If the answer is not found, say:
'I couldn't find the answer in the document.'
Context:
{context}
Question:
{question}
"""
  response=llm.generate_content(prompt)
  return response.text
with gr.Blocks() as demo:
    gr.Markdown("# 📄 RAG PDF Chatbot")

    pdf = gr.File(label="Upload PDF")
    upload_btn = gr.Button("Index PDF")
    status = gr.Textbox(label="Status")

    upload_btn.click(upload_pdf, inputs=pdf, outputs=status)

    question = gr.Textbox(label="Ask a Question")
    ask_btn = gr.Button("Ask")
    answer = gr.Textbox(label="Answer", lines=10)

    ask_btn.click(ask, inputs=question, outputs=answer)

demo.launch(debug=True)